# Validating the dataset

In [1]:
import json
import re
import requests
from urllib.parse import urlsplit
from IPython.display import Markdown, display
import numpy
import pandas as pd

data_path = "../data"

with open(f"{data_path}/quizzes.json", "r") as f:
    quizzes = json.load(f)

with open(f"{data_path}/timeout_exceptions.json", "r") as f:
    timeout_exceptions = json.load(f)

In [2]:
quizzes[0]

{'_id': 'KVicfAwDCYWseVLEXcQ1wgoc',
 'date': '2019-10-26',
 'source_url': 'https://www.morningbrew.com/issues/2019%2F10%2F26',
 'headlines': [{'text': 'Hitman hires hitman who hires hitman who hires hitman who hires hitman who tells police',
   'url': 'https://metro.co.uk/2019/10/23/hitman-hires-hitman-hires-hitman-hires-hitman-hires-hitman-tells-police-10971438/'},
  {'text': 'Man’s body brews its own beer after yeast take over his gut microbiome',
   'url': 'https://www.newscientist.com/article/2220432-mans-body-brews-its-own-beer-after-yeast-take-over-his-gut-microbiome/'},
  {'text': 'A parking space in Hong Kong just sold for almost $1 million',
   'url': 'https://www.bloomberg.com/news/articles/2019-10-24/hong-kong-car-parking-spot-sells-for-eye-watering-1-million'},
  {'text': 'Sanders staffer fired after accidentally ordering Uber helicopter to Queens rally',
   'url': 'https://media.giphy.com/media/l0NhWLdqrEy4hgeGc/giphy.gif'}],
 'answer': 'We made up the Bernie news.'}

In [2]:
print(f"Number of quizzes: {len(quizzes)}")

# with open(f"{data_path}/exceptions.json", "r") as f:
#     exceptions = json.load(f)
# print(f"Number of exceptions: {len(exceptions)}")

Number of quizzes: 337


Automated structural checks

In [7]:
with open(f"{data_path}/timeout_exceptions.json", "r") as f:
    timeout_exceptions = json.load(f)

"https://www.washingtonpost.com/nation/2020/03/04/jill-biden-joe-biden-protesters/" in timeout_exceptions["2020-03-07"]

True

In [10]:
date_search = re.compile(r'^\d{4}-\d{2}-\d{2}$')
dates_list = []
ids_list = []

def html_request(url):
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 404 or response.status_code == 460:
            print(f"{url} returned {response.status_code}")
    except requests.RequestException as e:
        print(f"{url} faild: {e}")

for quiz in quizzes:
    print(f"Checking quiz {quiz["date"]}")
    if not quiz["_id"]:
        print(f"{quiz["date"]} edition does not have an id")
    else:
        ids_list.append(quiz["_id"])
    if not date_search.search(quiz["date"]):
        print(f"{quiz["date"]} edition has invalid date")
    else:
        dates_list.append(quiz["date"])
    if (len(quiz["headlines"])) != 4:
        print(f"{quiz["date"]} edition does not have four headlines")
    for headline in quiz["headlines"]:
        url = headline["url"]
        if not headline["text"]:
            print(f"{quiz["date"]} edition has empty headline")
        if not url:
            print(f"{quiz["date"]} edition has empty URL")
        else:
            parts = urlsplit(url)
            if not parts.scheme:
                print(f"{url} edition has no scheme")
            if not parts.netloc:
                print(f"{url} edition has no domain")
            if not quiz["date"] in timeout_exceptions:
                html_request(url)
            else:
                if not url in timeout_exceptions[quiz["date"]]:
                    html_request(url)                
    if not any("giphy" in headline["url"] for headline in quiz["headlines"]):
        print(f"{quiz["date"]} edition has no GIF link")
    if not quiz["answer"]:
        print(f"{quiz["date"]} edition has empty answer")

ids_set = set(ids_list)
if len(ids_set) == len(ids_list):
    print("No duplicate ids detected")
else:
    print("Duplicate ids detected")
dates_set = set(dates_list)
if len(dates_set) == len(dates_list):
    print("No duplicate dates detected")
else:
    print("Duplicate dates detected")



Checking quiz 2019-10-26
Checking quiz 2019-11-02
Checking quiz 2019-11-09
Checking quiz 2019-11-16
Checking quiz 2019-11-23
Checking quiz 2019-12-07
Checking quiz 2019-12-14
Checking quiz 2019-12-21
Checking quiz 2020-01-04
Checking quiz 2020-01-18
2020-01-18 edition has no GIF link
Checking quiz 2020-01-25
Checking quiz 2020-02-01
Checking quiz 2020-02-08
Checking quiz 2020-02-15
Checking quiz 2020-02-22
Checking quiz 2020-02-29
Checking quiz 2020-03-07
Checking quiz 2020-03-14
Checking quiz 2020-03-21
https://ew.com/tv/coronavirus-masks-resident/ returned 460
Checking quiz 2020-03-28
2020-03-28 edition has no GIF link
Checking quiz 2020-04-04
Checking quiz 2020-04-18
Checking quiz 2020-04-25
https://people.com/movies/tom-hanks-gifts-corona-brand-typewriter-to-a-bullied-boy-named-corona/ returned 460
Checking quiz 2020-05-02
Checking quiz 2020-05-09
Checking quiz 2020-05-16
Checking quiz 2020-05-23
Checking quiz 2020-05-30
Checking quiz 2020-06-06
Checking quiz 2020-06-13
Checking qu

Automated anomaly detection

In [12]:
quizzes[1]

{'_id': 'z9gwGZ5GPFVVhEwb6cJdPxos',
 'date': '2019-11-02',
 'source_url': 'https://www.morningbrew.com/issues/2019%2F11%2F02',
 'headlines': [{'text': 'Leading maggot farmer to expand from Cape Town to California',
   'url': 'https://www.bloomberg.com/news/articles/2019-10-31/leading-maggot-farmer-to-expand-from-cape-town-to-california'},
  {'text': 'Riding wave of support for foie gras ban, de Blasio re-enters presidential race',
   'url': 'https://media.giphy.com/media/PoLEEuKQT7FnnJRiAA/giphy.gif'},
  {'text': 'Migrating Russian eagles run up huge data roaming charges',
   'url': 'https://www.bbc.com/news/world-europe-50180781'},
  {'text': 'Horsepower, literally: Finnish horse show runs on manure',
   'url': 'https://apnews.com/3ad8a6b9b1cb4100a9db1b2e406ad14d'}],
 'answer': 'Bill de Blasio is not re-entering the presidential race.'}

In [12]:
anomalous_characters = ["[", "]", "\n", "\r", "\t" "\xa0", "]("]
question_numbers = ["1.", "2.", "3.", "4."]


for quiz in quizzes:
    headlines = []
    print(f"Checking quiz {quiz["date"]}")
    for headline in quiz["headlines"]:
        text = headline["text"]
        headlines.append(text)
        url = headline["url"]
        if len(text) > 120:
            print(f"Long headline text detected in {quiz["date"]} edition: {text}")
        if len(text) < 40:
            print(f"Short headline text detected in {quiz["date"]} edition: {text}")
        if any(character in text for character in anomalous_characters):
            print(f"Anomalous characters detected in {quiz["date"]} edition headline: {repr(text)}")
        if any(text.startswith(question_number) for question_number in question_numbers):
            print(f"{quiz["date"]} edition headline starts with question number: {repr(text)}")
        if "https:" in text:
            print(f"{quiz["date"]} edition headline contains 'https:': {text}")
        if text.startswith(" "):
            print(f"{quiz["date"]} edition headline starts with whitespace: {text}")
        if text.endswith(" "):
            print(f"{quiz["date"]} edition headline ends with whitespace: {text}")
        if text.startswith("“") and text.endswith("”"):
            print(f"{quiz["date"]} edition headline starts and ends with curly quotation marks: {text}")
        if text.startswith("\"") and text.endswith("\""):
            print(f"{quiz["date"]} edition headline starts and ends with vertical quotation marks: {text}")
        if text.startswith("“") and text.endswith("\""):
            print(f"{quiz["date"]} edition headline starts with curly quotation marks and ends with vertical quotation marks: {text}")
        if text.startswith("\"") and text.endswith("”"):
            print(f"{quiz["date"]} edition headline starts with vertical quotation marks and ends with curly quotation marks: {text}")
        if "morningbrew" in url or "morning_brew" in url:
            print(f"{quiz["date"]} edition URL contains 'morningbrew' or 'morning_brew': {text}")
    if len(set(headlines)) != 4:
        print(f"{quiz["date"]} edition contains duplicate headlines: {quiz}")
    answer = quiz["answer"]
    if "answer" in answer:
        print(f"{quiz["date"]} edition answer contains 'answer': {quiz["answer"]}")
    if len(answer) > 110:
        print(f"Long answer text detected in {quiz["date"]} edition: {answer}")
    if len(answer) < 27:
        print(f"Short answer text detected in {quiz["date"]} edition: {answer}")

Checking quiz 2019-10-26
Checking quiz 2019-11-02
Checking quiz 2019-11-09
Checking quiz 2019-11-16
Short headline text detected in 2019-11-16 edition: Twinkies cereal is now in the works
Checking quiz 2019-11-23
Short headline text detected in 2019-11-23 edition: Why the world is running out of sand
Checking quiz 2019-12-07
Checking quiz 2019-12-14
Short headline text detected in 2019-12-14 edition: ‘My boss lets us book hangover days’
Checking quiz 2019-12-21
Checking quiz 2020-01-04
Checking quiz 2020-01-18
Long answer text detected in 2020-01-18 edition: The Men's Wearhouse story isn't real...but you should read the [Onion article](https://www.theonion.com/cash-strapped-men-s-wearhouse-now-offering-free-measure-1840982358?utm_source=morning_brew), it's hilarious.
Checking quiz 2020-01-25
Checking quiz 2020-02-01
Checking quiz 2020-02-08
Checking quiz 2020-02-15
Short headline text detected in 2020-02-15 edition: Psychedelics on the stock exchange?
Checking quiz 2020-02-22
Checking 

In [ ]:
for quiz in quizzes:
    headlines = []
    print(f"Checking quiz {quiz["date"]}")
    for headline in quiz["headlines"]:
        text = headline["text"]
        headlines.append(text)
    if len(set(headlines)) != 4:
        print(f"{quiz["date"]} edition contains duplicate headlines: {quiz}")

In [8]:
q_nums = ["1.", "2."]
string = "1. foo"
any(string.startswith(number) for number in q_nums)

True

In [ ]:
len(quizzes[0]["headlines"][0]["text"])

89

Manual sampling and validation

In [7]:
quizzes[1]


{'_id': 'z9gwGZ5GPFVVhEwb6cJdPxos',
 'date': '2019-11-02',
 'source_url': 'https://www.morningbrew.com/issues/2019%2F11%2F02',
 'headlines': [{'text': 'Leading maggot farmer to expand from Cape Town to California',
   'url': 'https://www.bloomberg.com/news/articles/2019-10-31/leading-maggot-farmer-to-expand-from-cape-town-to-california'},
  {'text': 'Riding wave of support for foie gras ban, de Blasio re-enters presidential race',
   'url': 'https://media.giphy.com/media/PoLEEuKQT7FnnJRiAA/giphy.gif'},
  {'text': 'Migrating Russian eagles run up huge data roaming charges',
   'url': 'https://www.bbc.com/news/world-europe-50180781'},
  {'text': 'Horsepower, literally: Finnish horse show runs on manure',
   'url': 'https://apnews.com/3ad8a6b9b1cb4100a9db1b2e406ad14d'}],
 'answer': 'Bill de Blasio is not re-entering the presidential race.'}

In [ ]:
for index in numpy.linspace(0, len(quizzes) - 1, num=30):
    index = round(index)
    display(quizzes[index]["date"])
    for headline in quizzes[index]["headlines"]:
        display(Markdown(headline["text"]))
        print(headline["url"])

In [26]:
response = requests.get("https://www.foodandwine.com/motorcycle-that-runs-on-beer-7498431", timeout=10)
response.status_code

460

In [43]:
with open(f"{data_path}/manual_quiz_additions.json", "r") as f:
    manual_additions = json.load(f)

for headline in manual_additions[3]["headlines"]:
        display(Markdown(headline["text"]))
        print(headline["url"])

Magma “heartbeat” is tearing continent apart, geologists discover

https://www.newsweek.com/geology-plate-tectonics-rift-mantle-plue-heartbeat-africa-afar-2089881


Board game lovers more likely to have successful marriages than karaoke fans

https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExOHdjYnJ1NjV3Mnh4YXVydGEydHA1cXhrc3RhZzdkZHN2bDFxOWU4YiZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/xT5LMsvDegy8V8C436/giphy.gif


Netflix’s “Liver King” arrested after “picking a fight” with Joe Rogan

https://www.hollywoodreporter.com/tv/tv-news/the-liver-king-netflix-brian-johnson-arrest-joe-rogan-1236298842/


BBC pledges to be “mindful” of spoilers after irate viewers complain about _Doctor Who_ reveal

https://deadline.com/2025/06/bbc-spoilers-doctor-who-complaints-1236442071/


Outstanding issues

In [4]:
with open(f"{data_path}/known_issues_outstanding.json", "r") as f:
    outstanding_issues = json.load(f)

# df = pd.DataFrame.from_dict(outstanding_issues, orient="index")

rows = []

for date, issue in outstanding_issues.items():
    for item in issue["edition_issues"] + issue["headline_issues"]:
        rows.append({
            "date": date,
            "type": item["type"],
            "description": item["description"]
        })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
display(df)

""
